<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which pages should a content team review first — refresh, investigate for an external cause, or leave alone — given limited weekly review capacity?

**Decision supported:** an SEO/content strategist deciding where to spend a fixed number of review hours each week. This is a *ranking/scoring* problem (Lane 2: Refresh / Content Opportunity Scoring), not plain classification — the decision-maker needs an ordered list, not a yes/no per page.

**Cost of a wrong call runs in both directions:**
- **False positive** (flagged, not actually declining): wasted editor time on a page that didn't need it.
- **False negative** (missed, actually declining): a real client keeps losing organic traffic that could have been caught early.

Both directions carry real cost, which is why this project is evaluated on **Precision@K** — of the top-K ranked pages, how many are actually worth reviewing? — always read against the test split's own base rate. A high Precision@K on an easy split is not the same achievement as one on a hard split; this notebook reports both every time.

**What this work is, and is not:** decision-support for a human reviewer with limited time. It does not predict Google's ranking algorithm, and it does not claim that refreshing a flagged page causes recovery — see Section 5, Limitations.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [9]:
%cd /content
!rm -rf flyrank-ml-internship
!git clone -q https://github.com/Ali-Shahrez/flyrank-ml-internship.git
%cd flyrank-ml-internship

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")
print(f"Distinct clients: {df['client_id'].nunique()}")
print(f"Columns: {df.shape[1]}")
print(f"Declining share (dataset-wide): {(df['trend_direction']=='down').mean():.3f}")

/content
/content/flyrank-ml-internship
Loaded 30,000 rows
Distinct clients: 32
Columns: 44
Declining share (dataset-wide): 0.542


**Primary dataset:** `data/raw/content_refresh_anonymized.csv` — 30,000 rows, one row per pseudonymized content item, 32 pseudonymized clients, 44 columns of trailing-90-day search/engagement metrics and content metadata. No client names, domains, URLs, titles, or raw search queries. `content_id` / `client_id` are hashed pseudonyms used only for grouping and client-holdout splits — never as model features.

**Warehouse contact (mechanics proof, not the modeling basis for this paper):** a separate data-contract exercise (`work/notebooks/w03_data_contract.ipynb`) queried the full `FlyRank/internship-warehouse` release on Hugging Face via DuckDB — one arbitrary month, March 2026: 9.8M rows, 55 of ~70 clients present that month, GSC availability 36.7% vs. GA4 availability 4.2%. That exercise proved the warehouse's grain, join keys, and availability-flag discipline, and reproduced this project's central leakage lesson on real warehouse rows (adding a same-window future column pushed ROC AUC from 0.601 to a suspicious 1.000). It was **not** used to train or validate the model reported in Sections 3–4 below — that result is scoped entirely to the CSV slice above. Moving the modeled result onto the warehouse, with a genuine future-window label, is the concrete next step named in Limitations.

**Time window:** the CSV is a single fixed 90-day trailing snapshot at export time — there is no separate feature window and label window inside it. This is precisely why the label used below is a same-window proxy rather than a genuine future outcome (Section 3).

**What was excluded, and why:**

| Excluded | Why |
|---|---|
| `trend_direction` / `trend_pct` | The label source. Present in the raw data, never used as a model feature — using them as features is the exact leakage case this project's methodology was built to catch. |
| `content_id` / `client_id` | Grouping and split keys only. Never fed to the model as features. |
| FlyRank's own product decision flags (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`) | Not shipped in this dataset at all, by design — nothing to accidentally learn from. |
| Warehouse keyword/content metadata beyond the CSV | Out of scope for the modeled result reported here; explored only in the separate data-contract mechanics exercise. |

**Public-safety confirmation:** no client names, domains, URLs, or raw queries appear anywhere in this notebook, its printed output, or its exported artifacts.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [10]:
import sys, os
sys.path.append(os.path.abspath('scripts'))
from ml_utils import precision_at_k, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

RANDOM_STATE = 42
MIN_IMPRESSIONS = 500

# --- Canonical client-holdout split (unique clients shuffled with a fixed seed) ---
client_series = df['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()

train_df = df[~test_mask].copy()
test_df  = df[test_mask].copy()

assert len(train_df) == 27675 and len(test_df) == 2325, "Split doesn't match the documented canonical numbers"
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,} | Test base rate: {(test_df['trend_direction']=='down').mean():.3f}")

# --- Eligibility gate, identical on both sides, same reason both times: rate metrics are noisy below a volume floor ---
train_eligible = train_df[train_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
test_eligible  = test_df[test_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
for f in (train_eligible, test_eligible):
    f['is_declining_label'] = (f['trend_direction'] == 'down').astype(int)

assert len(train_eligible) == 16108 and len(test_eligible) == 618
print(f"Train eligible: {len(train_eligible):,} | Test eligible: {len(test_eligible):,} | Test-eligible base rate: {test_eligible['is_declining_label'].mean():.3f}")
print(f"\nTest client counts (the concentration this notebook flags in Limitations):")
print(test_eligible['client_id'].value_counts())

Train rows: 27,675 | Test rows: 2,325 | Test base rate: 0.391
Train eligible: 16,108 | Test eligible: 618 | Test-eligible base rate: 0.519

Test client counts (the concentration this notebook flags in Limitations):
client_id
client_f74efabef1    598
client_d4735e3a26      9
client_4fc82b26ae      7
client_0b918943df      4
Name: count, dtype: int64


**Label:** `is_declining_label = (trend_direction == "down")`, where `trend_direction` is itself a threshold on `trend_pct` (last 30 days vs. prior 30 days of impressions, ±20%). This is a **proxy label** — a same-window calculation, not an observed future outcome. Every result in this notebook inherits that limitation; it is not treated as settled anywhere below.

**Baseline:** a transparent, *decline-blind* rule — percentile rank of `days_since_last_update` plus percentile rank of `impressions_90d`, summed. "Decline-blind" means the rule never looks at `trend_direction` or `trend_pct` to decide who's eligible or how they're ranked, so it can be compared honestly against a model trained to predict that same label.

**Model features:**
- *Numeric:* search volume, competition, CPC, word/char count, logged 90-day impressions/clicks/sessions/AI sessions, days with impressions/sessions, content age, days since last update, CTR, avg position, engagement rate, scroll rate, AI traffic %.
- *Categorical:* competition level, content type, main intent, age tier, freshness tier, word-count tier, impression tier, position tier.
- Missingness in keyword-context columns tracks `content_type` (confirmed: `feedly article` rows are 100% missing keyword data) — `has_*` flags are added *before* filling, so real missingness stays visible to the model instead of being silently zero-filled.

**Eligibility gate:** `impressions_90d >= 500`, applied identically to baseline and model, for the same reason both times — CTR and other rate-based metrics swing wildly below a minimum volume.

**Validation design — client-holdout split:** unique `client_id`s are shuffled with `np.random.default_rng(42)`, and the top 20% of clients (6 of 32) are held out entirely, so no client's pages appear in both train and test. This matters because pages from the same client can share patterns a model could memorize rather than genuinely learn from — the same failure mode that made an earlier hand-written rule look strong in-sample and collapse on unseen clients. After the eligibility gate, the test side is **618 rows across 4 clients, with one client contributing 598 of those (96.8%)** — printed above. This is a real limitation of an honest split on a 30K-row dataset, carried through every result below rather than hidden in Section 5.

**Leakage checks run:**
1. **Banned-column check** — confirmed `trend_direction`, `trend_pct`, `content_id`, `client_id` never enter the feature matrix (asserted in code below).
2. **Deliberate-leak calibration** — adding `trend_pct` back in as a feature pushed the honest model's top-feature importance from 0.107 (a 1.3× gap to the #2 feature) to 0.836 (a 42× gap), and Precision@K to a suspicious 1.000. This gives a concrete "what a real leak looks like in this pipeline" signature to compare the honest model against.
3. **Warehouse-level replication** — deliberately adding a same-window future column to a separate warehouse-based model reproduced the identical signature (ROC AUC 0.601 → 1.000), confirming the leakage mechanism isn't specific to this one CSV.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

MISSINGNESS_TRACKED_COLS = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc']

def add_derived_and_flags(frame):
    frame = frame.copy()
    for col in MISSINGNESS_TRACKED_COLS:
        frame[f'has_{col}'] = frame[col].notna().astype(int)
    frame['log_impressions_90d'] = np.log1p(frame['impressions_90d'])
    frame['log_clicks_90d']      = np.log1p(frame['clicks_90d'])
    frame['log_sessions_90d']    = np.log1p(frame['sessions_90d'])
    frame['log_ai_sessions_90d'] = np.log1p(frame['ai_sessions_90d'])
    return frame

def build_features(frame, numeric_cols, categorical_cols, flag_cols):
    numeric = frame[numeric_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    flags = frame[flag_cols]
    categorical = frame[categorical_cols].fillna('unknown').astype(str)
    dummies = pd.get_dummies(categorical, prefix=categorical_cols, dtype=float)
    return pd.concat([numeric.reset_index(drop=True), flags.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

train_eligible = add_derived_and_flags(train_eligible)
test_eligible  = add_derived_and_flags(test_eligible)

numeric_cols = [c for c in MODEL_NUMERIC_FEATURES if c in train_eligible.columns]
categorical_cols = [c for c in MODEL_CATEGORICAL_FEATURES if c in train_eligible.columns]
flag_cols = [f'has_{c}' for c in MISSINGNESS_TRACKED_COLS]

X_train = build_features(train_eligible, numeric_cols, categorical_cols, flag_cols)
X_test  = build_features(test_eligible,  numeric_cols, categorical_cols, flag_cols)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train_eligible['is_declining_label']
y_test  = test_eligible['is_declining_label']

assert not any(c in X_train.columns for c in ['trend_direction', 'trend_pct', 'content_id', 'client_id']), \
    "Leakage check failed"
print("Leakage check passed: no trend_direction / trend_pct / content_id / client_id in the feature matrix.")

# --- Baseline, recomputed on the test-eligible split (same formula as Section 3) ---
test_eligible['baseline_score'] = (
    test_eligible['days_since_last_update'].rank(pct=True) +
    test_eligible['impressions_90d'].rank(pct=True)
)
baseline_p20 = precision_at_k(y_test, test_eligible['baseline_score'], k=20)
baseline_p50 = precision_at_k(y_test, test_eligible['baseline_score'], k=50)

# --- Logistic Regression ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)
logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_p20 = precision_at_k(y_test, logreg_scores, k=20)
logreg_p50 = precision_at_k(y_test, logreg_scores, k=50)

# --- Random Forest ---
rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                             min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
rf_test_scores = rf.predict_proba(X_test)[:, 1]
rf_p20 = precision_at_k(y_test, rf_test_scores, k=20)
rf_p50 = precision_at_k(y_test, rf_test_scores, k=50)

# --- Overfitting sanity check: train-set score on the same Random Forest ---
train_scores_rf = rf.predict_proba(X_train)[:, 1]
train_p20_rf = precision_at_k(y_train, train_scores_rf, k=20)
train_p50_rf = precision_at_k(y_train, train_scores_rf, k=50)

comparison_table = pd.DataFrame({
    'Method': ['Baseline (decline-blind rule)', 'Logistic Regression', 'Random Forest', 'Test base rate'],
    'Precision@20': [baseline_p20, logreg_p20, rf_p20, y_test.mean()],
    'Precision@50': [baseline_p50, logreg_p50, rf_p50, y_test.mean()],
})
print(f"All rows evaluated on the identical honest test-eligible split, n={len(y_test)}:\n")
print(comparison_table.round(3).to_string(index=False))
print(f"\nRandom Forest on its OWN training rows (in-sample, not a validated number): P@20={train_p20_rf:.3f}  P@50={train_p50_rf:.3f}")

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 Random Forest feature importances (honest model):")
print(importances.head(10).round(4))

Leakage check passed: no trend_direction / trend_pct / content_id / client_id in the feature matrix.
All rows evaluated on the identical honest test-eligible split, n=618:

                       Method  Precision@20  Precision@50
Baseline (decline-blind rule)         0.250         0.240
          Logistic Regression         0.550         0.480
                Random Forest         0.850         0.860
               Test base rate         0.519         0.519

Random Forest on its OWN training rows (in-sample, not a validated number): P@20=1.000  P@50=1.000

Top 10 Random Forest feature importances (honest model):
content_age_days      0.1071
avg_position          0.0831
log_clicks_90d        0.0770
scroll_rate           0.0770
ctr                   0.0650
days_with_sessions    0.0612
char_count            0.0453
word_count            0.0445
age_tier_365+         0.0425
log_sessions_90d      0.0416
dtype: float64


Reading this against the base rate (0.519), not just against each other:

| Method | P@20 | P@50 | Beats the 0.519 base rate? |
|---|---:|---:|---|
| Baseline (decline-blind rule) | 0.250 | 0.240 | No — well below chance |
| Logistic Regression | 0.550 | 0.480 | Barely at P@20 / no at P@50 |
| Random Forest | 0.850 | 0.860 | Yes, clearly, at both K |

The baseline is *worse than random guessing* on this split — a real, meaningful negative result, not a bug. Logistic Regression clears Precision@20 by only 0.031 over base rate and **loses to chance** at Precision@50. Random Forest is the only method that beats both the baseline and chance substantially at both K values.

**But the 0.86 number needs a caveat, not a headline.** Random Forest's own training-set Precision@50 = 1.000 (printed above) is a documented overfitting signature. A `min_samples_leaf` regularization sweep (run separately, see `work/notebooks/w05_model.ipynb`) showed test Precision@20/50 swinging noisily between roughly 0.65 and 0.86 as tree complexity was pulled back, while train Precision@50 barely moved — the signature of a test set too small (618 rows, 96.8% one client) to pin a Precision@K down to two decimal places. The honest range to report is **"0.65–0.86 depending on regularization, consistently above the 0.519 base rate"** — a real, repeatable direction of result — not a single confident 0.86. Section 5 explains why this test set can't resolve the number more precisely, and what would fix it.

**A second, different kind of uncertainty check.** The regularization sweep above asks "does the number move if the model changes?" This bootstrap asks a different question: "does the number move if the *sample* changes?" — holding the trained model and its scores fixed, and resampling which of the 618 test rows are drawn, 2,000 times.

In [12]:
# --- Bootstrap: how much does Precision@K wobble just from which test rows we happened to draw? ---
# This is a DIFFERENT question from the regularization sweep above: that varies the model,
# this varies the sample. Model and scores are held fixed; only which of the 618 rows are
# resampled (with replacement) changes each iteration.
N_BOOT = 2000
boot_rng = np.random.default_rng(RANDOM_STATE)
y_test_arr = y_test.to_numpy()
n = len(y_test_arr)
p20_samples = np.empty(N_BOOT)
p50_samples = np.empty(N_BOOT)

for i in range(N_BOOT):
    idx = boot_rng.integers(0, n, size=n)
    p20_samples[i] = precision_at_k(y_test_arr[idx], rf_test_scores[idx], 20)
    p50_samples[i] = precision_at_k(y_test_arr[idx], rf_test_scores[idx], 50)

p20_lo, p20_hi = np.percentile(p20_samples, [5, 95])
p50_lo, p50_hi = np.percentile(p50_samples, [5, 95])
base_rate = y_test_arr.mean()

print(f"Bootstrap (N={N_BOOT}, resampling the {n}-row test set with replacement):")
print(f"P@20: median={np.median(p20_samples):.3f}  90% interval=[{p20_lo:.3f}, {p20_hi:.3f}]")
print(f"P@50: median={np.median(p50_samples):.3f}  90% interval=[{p50_lo:.3f}, {p50_hi:.3f}]")
print(f"\nShare of resamples where P@20 > base rate ({base_rate:.3f}): {(p20_samples > base_rate).mean():.3f}")
print(f"Share of resamples where P@50 > base rate ({base_rate:.3f}): {(p50_samples > base_rate).mean():.3f}")
print("\nCaveat: this interval only captures row-level sampling noise within these 618 rows.")
print("It cannot inject client diversity that isn't there -- 96.8% of this test set is one")
print("client, so the interval still describes that client's content, not the wider portfolio.")

Bootstrap (N=2000, resampling the 618-row test set with replacement):
P@20: median=0.850  90% interval=[0.700, 0.950]
P@50: median=0.860  90% interval=[0.760, 0.940]

Share of resamples where P@20 > base rate (0.519): 0.999
Share of resamples where P@50 > base rate (0.519): 1.000

Caveat: this interval only captures row-level sampling noise within these 618 rows.
It cannot inject client diversity that isn't there -- 96.8% of this test set is one
client, so the interval still describes that client's content, not the wider portfolio.


## 5. Limitations

*What this work cannot claim.*

In [13]:
tie_mass_pct = (test_eligible['days_since_last_update'] == test_eligible['days_since_last_update'].mode()[0]).mean()
dominant_client_share = test_eligible['client_id'].value_counts(normalize=True).max()
print(f"Most common days_since_last_update value in the test-eligible slice: {test_eligible['days_since_last_update'].mode()[0]}")
print(f"Share of test-eligible rows sharing that exact value: {tie_mass_pct:.3f}")
print(f"Share of test-eligible rows from the single dominant client: {dominant_client_share:.3f}")

Most common days_since_last_update value in the test-eligible slice: 20
Share of test-eligible rows sharing that exact value: 0.879
Share of test-eligible rows from the single dominant client: 0.968


1. **Same-window proxy label, not a future outcome.** `is_declining_label` compares two halves of an already-completed 90-day window against each other. It has never been validated against what actually happened to a page afterward. The concrete next step (out of scope for this pass) is a genuine future-window label on the warehouse: prior 90 days of features predicting decline over the next 30.
2. **The out-of-sample test set is effectively one client.** Printed above: 96.8% of test-eligible rows belong to a single client. The 0.65–0.86 Precision@K range in Section 4 describes that client's content specifically — extending that confidence to the rest of the portfolio, or to any other client, is not supported by this test design.
3. **A structural ceiling, not a fixable bug.** The honest feature set cannot separate a currently-declining page from a currently-recovering one that happens to share the same weak-position / low-CTR / stale-update profile. The one feature that would resolve the ambiguity — recent trend direction — is exactly the column the leakage audit in Section 3 correctly excludes from the model.
4. **A synthetic-data tie-mass artifact affects the staleness signal.** Confirmed above: 87.9% of this test-eligible slice shares one exact `days_since_last_update` value (20); a *different* single value (104) accounts for 38.6% of the full 16,726-row eligible population (documented in `work/notebooks/w04_baseline_score.ipynb`). Different value, same shape — almost certainly a batch artifact of the anonymization/synthesis process, not real-world staleness variation. It degrades rank-based scoring at depth (why the baseline's Precision@50 is worse than its Precision@20) and weakens the `STALE_UPDATE` reason code used in the playbook (Section 6).
5. **Selection and lookalike risk in the label itself.** A drop in impressions can also be consolidation (a sibling page absorbing demand), seasonality, or a SERP/AI-feature click-loss — none of which this label or model can distinguish from a decline a refresh would actually fix.
6. **No causal claim.** Nothing in this notebook shows that refreshing a flagged page *causes* recovery. That would need an experiment — e.g. staggered refresh timing with a matched control — which was not run here. (This is the same standard applied when auditing FlyRank's own published refresh-timing finding in `work/notebooks/w06_validation_audit.ipynb`, which has the same gap.)
7. **Not validated on the full warehouse.** The modeled result above is scoped entirely to the 30K-row anonymized CSV. A separate exercise queried the real ~79M-row warehouse only to prove grain, joins, and leakage mechanics on one arbitrary month — that exercise does not extend this model's validity.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [14]:
train_eligible['rf_score'] = rf.predict_proba(X_train)[:, 1]
test_eligible['rf_score']  = rf.predict_proba(X_test)[:, 1]
train_eligible['validation_status'] = 'in_sample_fit'
test_eligible['validation_status']  = 'out_of_sample'

queue = pd.concat([train_eligible, test_eligible], ignore_index=True)

queue['MODEL_DECLINE_RISK'] = queue['rf_score'] >= 0.65
queue['RANKING_SLIPPED'] = (queue['avg_position'] > 20)
queue['RANKING_INTACT_DEMAND_DROP'] = (queue['avg_position'] > 0) & (queue['avg_position'] <= 10)
queue['WEAK_CTR_FOR_POSITION'] = (queue['impressions_90d'] >= 500) & (queue['avg_position'] > 0) & (queue['avg_position'] <= 20) & (queue['ctr'] < 0.5)
queue['LOW_ENGAGEMENT'] = (queue['sessions_90d'] >= 30) & ((queue['engagement_rate'] < 30) | (queue['scroll_rate'] < 30))
queue['STALE_UPDATE'] = queue['freshness_tier'].isin(['31-90', '91-180'])

def assign_action(row):
    if not row['MODEL_DECLINE_RISK']:
        return 'no_action', []
    codes = []
    if row['RANKING_SLIPPED']:
        codes.append('RANKING_SLIPPED'); action = 'refresh_content'
    elif row['RANKING_INTACT_DEMAND_DROP']:
        codes.append('RANKING_INTACT_DEMAND_DROP'); action = 'investigate_external'
    elif row['WEAK_CTR_FOR_POSITION']:
        codes.append('WEAK_CTR_FOR_POSITION'); action = 'review_metadata'
    elif row['LOW_ENGAGEMENT']:
        codes.append('LOW_ENGAGEMENT'); action = 'review_onpage_engagement'
    else:
        action = 'monitor_closely'
    if row['STALE_UPDATE']:
        codes.append('STALE_UPDATE')
    return action, codes

res = queue.apply(assign_action, axis=1, result_type='expand')
queue['action'] = res[0]
queue['reason_codes'] = res[1].apply(lambda cs: ','.join(['MODEL_DECLINE_RISK'] + cs))
queue.loc[queue['action'] == 'no_action', 'reason_codes'] = ''

status_order = {'out_of_sample': 0, 'in_sample_fit': 1}
queue['status_rank'] = queue['validation_status'].map(status_order)
queue_ranked = queue.sort_values(by=['status_rank', 'rf_score'], ascending=[True, False]).drop(columns='status_rank').reset_index(drop=True)

oos = queue_ranked[queue_ranked['validation_status'] == 'out_of_sample']
action_mix_oos = oos['action'].value_counts()
print("Action counts, OUT-OF-SAMPLE tier only (n=618, the trustworthy tier):")
print(action_mix_oos)

flagged_queue = queue_ranked[queue_ranked['action'] != 'no_action']
top3_share = flagged_queue['client_id'].value_counts().head(3).sum() / len(flagged_queue)
print(f"\nTotal flagged (actionable) rows, full 16,726-row eligible population: {len(flagged_queue):,}")
print(f"Distinct clients represented: {flagged_queue['client_id'].nunique()}")
print(f"Top-3 client share of flagged rows: {top3_share:.3f}")

stale_secondary = queue_ranked[queue_ranked['MODEL_DECLINE_RISK']]['reason_codes'].str.contains('STALE_UPDATE').sum()
print(f"Rows carrying STALE_UPDATE as a secondary reason code: {stale_secondary:,}")

below_gate = (df['impressions_90d'] < MIN_IMPRESSIONS).sum()
print(f"Pages below the eligibility gate (no recommendation at all): {below_gate:,} of {len(df):,} ({below_gate/len(df):.1%})")

cols = ['content_id', 'client_id', 'rf_score', 'validation_status', 'action', 'reason_codes', 'avg_position', 'ctr', 'impressions_90d']
queue_ranked[cols].head(10)

Action counts, OUT-OF-SAMPLE tier only (n=618, the trustworthy tier):
action
no_action               414
review_metadata          83
investigate_external     76
refresh_content          45
Name: count, dtype: int64

Total flagged (actionable) rows, full 16,726-row eligible population: 4,290
Distinct clients represented: 23
Top-3 client share of flagged rows: 0.675
Rows carrying STALE_UPDATE as a secondary reason code: 1,787
Pages below the eligibility gate (no recommendation at all): 13,274 of 30,000 (44.2%)


,content_id,client_id,rf_score,validation_status,action,reason_codes,avg_position,ctr,impressions_90d
0,content_6e792cf3ce56,client_f74efabef1,0.780279,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",29.1,0.06,4908
1,content_0cf67ec37ab8,client_f74efabef1,0.776617,out_of_sample,investigate_external,"MODEL_DECLINE_RISK,RANKING_INTACT_DEMAND_DROP",3.0,0.00,767
2,content_331182ca4cae,client_f74efabef1,0.751619,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",35.9,0.00,3026
3,content_52b1c884e871,client_f74efabef1,0.750648,out_of_sample,review_metadata,"MODEL_DECLINE_RISK,WEAK_CTR_FOR_POSITION",15.1,0.15,682
4,content_818c81a114e4,client_f74efabef1,0.745253,out_of_sample,review_metadata,"MODEL_DECLINE_RISK,WEAK_CTR_FOR_POSITION",15.1,0.00,1243
5,content_d603c0b7de2e,client_f74efabef1,0.744184,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",23.3,0.00,554
6,content_89f63af999b1,client_f74efabef1,0.743108,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",28.6,0.00,901
7,content_1df2423841db,client_f74efabef1,0.738269,out_of_sample,investigate_external,"MODEL_DECLINE_RISK,RANKING_INTACT_DEMAND_DROP",9.4,0.00,1343
8,content_fded62af1b93,client_f74efabef1,0.736958,out_of_sample,investigate_external,"MODEL_DECLINE_RISK,RANKING_INTACT_DEMAND_DROP",7.1,0.00,547
9,content_00603b0349b4,client_f74efabef1,0.732427,out_of_sample,refresh_content,"MODEL_DECLINE_RISK,RANKING_SLIPPED",25.6,0.09,1076


**Sort order is itself a claim.** The queue is sorted by `validation_status` first (out-of-sample before in-sample-fit), score second — so every row a reviewer sees first is genuinely out-of-sample, never a row the model was fit on.

**Reading the numbers above:**
- Out-of-sample action mix: mostly `no_action` (414 of 618), with `review_metadata`, `investigate_external`, and `refresh_content` splitting the rest. Two possible actions never fire in this tier — reported as-is, not treated as evidence those categories are rare in general, given this tier is ~97% one client.
- Across the full 16,726-row eligible population, **4,290 rows are flagged** across **23 distinct clients** — but the top 3 clients account for **67.5%** of flagged rows, so treat scores for any client outside that concentration, or outside the 618-row out-of-sample tier, as less proven regardless of label.
- **1,787 rows** carry `STALE_UPDATE` as a secondary tag — never the action-driving reason on its own, and per Section 5 this reason code leans on a value that's likely a synthetic-data artifact, so it is the weakest reason code in the set.
- **44.2% of the full 30,000-row dataset** (13,274 pages) falls below the eligibility gate and receives *no recommendation at all* — not scored as low-priority, simply outside this tool's coverage. A separate, lower-volume-appropriate process would be needed for them.

**Who uses this, and how:** a content strategist with limited weekly review time, using the queue to decide what to open first — never as an auto-apply system. Every row is a suggestion for a human to check against the page itself, per the reason codes attached, before taking any action.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [15]:
import sys
sys.path.append(os.path.abspath('scripts'))
from ml_utils import simple_svg_bar_chart
from pathlib import Path

os.makedirs('work/figures', exist_ok=True)

simple_svg_bar_chart(
    title="Action mix — out-of-sample (validated) rows only, n=618",
    labels=list(action_mix_oos.index), values=list(action_mix_oos.values),
    path=Path('work/figures/action_mix_out_of_sample.svg'),
)

top_importances = importances.head(10)
simple_svg_bar_chart(
    title="Top 10 Random Forest feature importances (honest model)",
    labels=list(top_importances.index), values=list(top_importances.values),
    path=Path('work/figures/rf_feature_importance.svg'),
    color="#3E7CB1",
)

simple_svg_bar_chart(
    title="Precision@20 vs baseline & base rate (honest test split, n=618)",
    labels=list(comparison_table['Method']), values=list(comparison_table['Precision@20']),
    path=Path('work/figures/precision_at_20_comparison.svg'),
    color="#D9822B",
)
simple_svg_bar_chart(
    title="Precision@50 vs baseline & base rate (honest test split, n=618)",
    labels=list(comparison_table['Method']), values=list(comparison_table['Precision@50']),
    path=Path('work/figures/precision_at_50_comparison.svg'),
    color="#D9822B",
)

print("Saved:")
for f in sorted(os.listdir('work/figures')):
    print(" -", f)

# Export the queue + a metrics summary for the paper page to reference
queue_ranked[cols + ['engagement_rate', 'scroll_rate', 'days_since_last_update']].to_csv('work/outputs/action_playbook_queue.csv', index=False) if os.path.isdir('work/outputs') else os.makedirs('work/outputs', exist_ok=True)
queue_ranked[cols + ['engagement_rate', 'scroll_rate', 'days_since_last_update']].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print("\nSaved work/outputs/action_playbook_queue.csv")

Saved:
 - action_mix_out_of_sample.svg
 - precision_at_20_comparison.svg
 - precision_at_50_comparison.svg
 - rf_feature_importance.svg

Saved work/outputs/action_playbook_queue.csv


Four charts feed the deployed paper directly:

| File | Used in paper section |
|---|---|
| `work/figures/precision_at_20_comparison.svg` | Results — Precision@20 |
| `work/figures/precision_at_50_comparison.svg` | Results — Precision@50 |
| `work/figures/rf_feature_importance.svg` | Results / interpretation |
| `work/figures/action_mix_out_of_sample.svg` | Ranked recommendations |

Each chart gets a one-sentence takeaway written directly under it on the deployed page (per the `writing-research-papers` skill) rather than a bare caption — the point is that someone skimming headings + chart takeaways alone should come away with the correct finding and its limits.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.